# L22 · 瞬间的斜率 — 导数

**目标**：导数就是「输入动一点，输出动多少」——曲线的斜率。

**为什么对 Aurora 重要**：`numeric_derivative` 是验证反向传播正确性的标准工具——用数值梯度和解析梯度逐元素比对，确认 MFCC、梅尔滤波器等模块的求导实现没有错误。

## 本课剧情：用放大镜看变化

中心差分把导数变成可以计算的操作：在 x 左右各取一个点，用输出差除以间距 2h。h 取 1e-5 时，误差已在 1e-10 量级——比单侧差分精确两个数量级。本节实现 `numeric_derivative`，并对照 sin(x) 的解析导数 cos(x) 验证结果。

## 1. 用数值方法近似导数（中心差分）

`f'(x) ≈ (f(x+h) - f(x-h)) / (2h)`，h 取很小的数。

中心差分之所以优于前向差分 `(f(x+h) - f(x)) / h`，在于截断误差阶数不同。前向差分用 x 和 x+h 两点连线，斜率偏向右侧，误差是 O(h)。中心差分关于 x 对称取点，泰勒展开时一阶误差项正负抵消，只剩 h² 项，误差是 O(h²)。h=1e-5 时，前向差分误差约 1e-5，中心差分误差约 1e-10——差两个数量级。

h 并非越小越好：当 h 缩到 1e-12 以下，`f(x+h) - f(x-h)` 两个几乎相等的浮点数相减，有效位数从 16 位降到 4 位左右（catastrophic cancellation），误差反而急剧上升。最优 h 在 1e-5 到 1e-7 之间，此处截断误差与舍入误差平衡，总误差最小。

## 实验入口：用数值变化观察函数

下面的实验在 x=1.0 处计算 sin(x) 的数值导数，关注 `h` 缩小时误差如何先降后升，找到精度最优的步长区间。

In [ ]:
import numpy as np
f = lambda x: x**2
h = 1e-5
x = 3.0
print('f(x)=x^2 在 x=3 的斜率 ≈', (f(x+h)-f(x-h))/(2*h), ' (真值 2x=6)')

## 动手观察：变化率不是一句口号

在 x=π/4 处运行一次，打印 `(f(x+h) - f(x-h)) / (2h)` 和 cos(π/4)=0.7071，确认两者误差在 1e-10 以内。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

在 [0, 2π] 上取 10 个点，逐点打印数值导数与 cos(x) 的差值，观察误差在整个区间上都稳定在 1e-10 量级。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `numeric_derivative(f, x, h=1e-5)`

**推理路线**：
1. 函数签名：接受 `f`（任意可调用）、`x`（float，求导点）、`h`（扰动步长，默认 1e-5）。
2. 计算 `f(x+h)` 和 `f(x-h)`——各向右/左扰动 h，共两次函数求值。
3. 两者相减后除以 `2*h`，得到中心差分近似，返回单个 float。

**参考输入输出**：f=sin, x=0, h=1e-5 → `(sin(1e-5) - sin(-1e-5))/(2e-5)` ≈ `(1e-5 - (-1e-5))/(2e-5)` = 1.0 = cos(0) ✓

<details><summary>点击查看参考实现</summary>

```python
def numeric_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)
```

</details>

### 写代码前，先把变量表补完整

写 `numeric_derivative` 前明确三件事：
- 输入：`f`（可调用函数）、`x`（求导点）、`h`（扰动步长，默认 1e-5）
- 关键步骤：计算 `f(x+h)` 和 `f(x-h)`，相减后除以 `2h`
- 返回：单个浮点数，表示 f 在 x 处的斜率近似值

In [ ]:
def numeric_derivative(f, x, h=1e-5):
    # ✏️ TODO: 中心差分近似导数
    return None  # ← 改这里


In [ ]:
assert abs(numeric_derivative(lambda x: x**2, 3.0) - 6.0) < 1e-3
assert abs(numeric_derivative(np.sin, 0.0) - 1.0) < 1e-3   # sin'(0)=cos(0)=1
print('✅ 通过：你会数值求导了。')

## 3. 记几个常见导数（背下来省事）

| f(x) | f'(x) |
|---|---|
| xⁿ | n·xⁿ⁻¹ |
| eˣ | eˣ |
| sin x | cos x |
| ln x | 1/x |

**🔗 Aurora 连接**：`numeric_derivative` 在 Aurora 梯度检验（`src/aurora/audio/grad_check.py`）中直接调用——将 MFCC、梅尔滤波器等模块的解析梯度与数值梯度逐元素比对，差异超过 1e-5 即判定求导实现有误。下一节（`c2_gradients.ipynb`）把单点斜率推广到多变量：沿每个参数维度重复一次 `numeric_derivative`，拼成梯度向量。

In [ ]:
def f(x):
    return (x - 2)**2 + 1

x = -2.0
lr = 0.2
for step in range(8):
    grad = 2 * (x - 2)
    print(f'step={step} | x={x:6.3f} | f(x)={f(x):6.3f} | grad={grad:6.3f}')
    x = x - lr * grad
print('你刚刚让 x 一步步靠近了最低点 2。')


## 参数实验：h 的最优区间

把 `h` 从 1e-1 逐步减小到 1e-15，每步打印 `abs(numeric_derivative(np.sin, 0, h) - 1.0)`，观察误差先减后增的 U 形曲线，找到最优步长约 1e-5~1e-7。

- **h 偏大（1e-1 到 1e-3）**：截断误差主导，误差约 h²/6·f'''(x)，随 h 缩小以平方速率下降。
- **h 最优（1e-5 到 1e-7）**：截断误差与浮点舍入误差平衡，总误差最小约 1e-10。
- **h 过小（1e-12 以下）**：`f(x+h) - f(x-h)` 两个近似相等的数相减，有效位数从 16 位骤降至 4 位以下，误差反而随 h 减小而上升。

In [ ]:
def f(x):
    return (x - 2)**2 + 1

x = -2.0
lr = 0.2
for step in range(8):
    grad = 2 * (x - 2)
    print(f'step={step} | x={x:6.3f} | f(x)={f(x):6.3f} | grad={grad:6.3f}')
    x = x - lr * grad
print('观察：x 正在靠近最低点 2。')


## 本课收束

现在可以用 `numeric_derivative(f, x, h=1e-5)` 在任意点计算可调用函数的斜率，输出单个浮点数，误差量级 1e-10。这个函数在 Aurora 梯度检验中直接使用：将损失对权重的解析梯度与数值梯度逐元素比对，差异超过 1e-5 即报错。`h` 的最优范围在 1e-5 附近，过大则截断误差主导，过小则浮点舍入误差主导。下一节（c2_gradients.ipynb）把这个操作沿每个参数维度重复，得到多变量函数的梯度向量。

In [ ]:
# 小检查：把“变化一点点”写成可以观察的数字
import numpy as np

def f(x):
    return x**2

x = 3.0
for h in [1.0, 0.1, 0.01, 0.001]:
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'h={h:6.3f} -> 近似斜率 {slope:.6f}')
print('理论值 2*x =', 2*x)
